In [1]:
DATA_DIR = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction"

In [2]:
import os

DATA_DIR = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction"

for file in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, file)

    if os.path.isfile(path):
        size_gb = os.path.getsize(path) / (1024 ** 3)
        print(f"{file:30s} {size_gb:.3f} GB")

sample_submission.csv          0.000 GB
GCP-Coupons-Instructions.rtf   0.000 GB
train.csv                      5.306 GB
test.csv                       0.001 GB


In [3]:
import pandas as pd

TRAIN_PATH = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/train.csv"

sample = pd.read_csv(TRAIN_PATH, nrows=5)

print("Shape of sampled data:", sample.shape)
print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 rows:")
display(sample)

Shape of sampled data: (5, 8)

Columns:
['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

Data types:
key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

First 5 rows:


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21 UTC,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16 UTC,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00 UTC,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42 UTC,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00 UTC,-73.968095,40.768008,-73.956655,40.783762,1


In [4]:
row_count = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):
    row_count += len(chunk)
    print(f"Processed: {row_count:,} rows")

print(f"\nTotal rows: {row_count:,}")

Processed: 1,000,000 rows
Processed: 2,000,000 rows
Processed: 3,000,000 rows
Processed: 4,000,000 rows
Processed: 5,000,000 rows
Processed: 6,000,000 rows
Processed: 7,000,000 rows
Processed: 8,000,000 rows
Processed: 9,000,000 rows
Processed: 10,000,000 rows
Processed: 11,000,000 rows
Processed: 12,000,000 rows
Processed: 13,000,000 rows
Processed: 14,000,000 rows
Processed: 15,000,000 rows
Processed: 16,000,000 rows
Processed: 17,000,000 rows
Processed: 18,000,000 rows
Processed: 19,000,000 rows
Processed: 20,000,000 rows
Processed: 21,000,000 rows
Processed: 22,000,000 rows
Processed: 23,000,000 rows
Processed: 24,000,000 rows
Processed: 25,000,000 rows
Processed: 26,000,000 rows
Processed: 27,000,000 rows
Processed: 28,000,000 rows
Processed: 29,000,000 rows
Processed: 30,000,000 rows
Processed: 31,000,000 rows
Processed: 32,000,000 rows
Processed: 33,000,000 rows
Processed: 34,000,000 rows
Processed: 35,000,000 rows
Processed: 36,000,000 rows
Processed: 37,000,000 rows
Processed:

In [5]:
sample_df = pd.read_csv(
    TRAIN_PATH,
    nrows=100_000
)

print("Shape:", sample_df.shape)

print("\nMissing values:")
print(sample_df.isna().sum())

print("\nData types:")
print(sample_df.dtypes)

print("\nNumerical summary:")
display(sample_df.describe())

Shape: (100000, 8)

Missing values:
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

Data types:
key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

Numerical summary:


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,11.354652,-72.494682,39.914481,-72.490967,39.919053,1.673820
std,9.716777,10.693934,6.225686,10.471386,6.213427,1.300171
min,-44.900000,-736.550000,-74.007670,-84.654241,-74.006377,0.000000
25%,6.000000,-73.992041,40.734996,-73.991215,40.734182,1.000000
50%,8.500000,-73.981789,40.752765,-73.980000,40.753243,1.000000
75%,12.500000,-73.966982,40.767258,-73.963433,40.768166,2.000000
max,200.000000,40.787575,401.083332,40.851027,404.616667,6.000000


In [6]:
checks = {
    "negative_fare": sample_df["fare_amount"] <= 0,

    "invalid_pickup_longitude": ~sample_df["pickup_longitude"].between(-75, -72),

    "invalid_pickup_latitude": ~sample_df["pickup_latitude"].between(40, 42),

    "invalid_dropoff_longitude": ~sample_df["dropoff_longitude"].between(-75, -72),

    "invalid_dropoff_latitude": ~sample_df["dropoff_latitude"].between(40, 42),

    "invalid_passenger_count": sample_df["passenger_count"] <= 0,
}

for name, condition in checks.items():
    print(f"{name:30s}: {condition.sum():,} rows ({condition.mean()*100:.2f}%)")

negative_fare                 : 12 rows (0.01%)
invalid_pickup_longitude      : 1,995 rows (1.99%)
invalid_pickup_latitude       : 1,999 rows (2.00%)
invalid_dropoff_longitude     : 1,983 rows (1.98%)
invalid_dropoff_latitude      : 1,983 rows (1.98%)
invalid_passenger_count       : 366 rows (0.37%)


In [7]:
invalid_mask = (
    checks["negative_fare"]
    | checks["invalid_pickup_longitude"]
    | checks["invalid_pickup_latitude"]
    | checks["invalid_dropoff_longitude"]
    | checks["invalid_dropoff_latitude"]
    | checks["invalid_passenger_count"]
)

print("Total invalid rows:", invalid_mask.sum())
print("Percentage of sample:", f"{invalid_mask.mean()*100:.2f}%")

Total invalid rows: 2479
Percentage of sample: 2.48%


In [8]:
invalid_rows = sample_df[invalid_mask]

print("Invalid rows shape:", invalid_rows.shape)

display(invalid_rows.head(20))

print("\nSummary of invalid rows:")
display(invalid_rows.describe())

Invalid rows shape: (2479, 8)


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
11,2012-12-24 11:24:00.00000098,5.5,2012-12-24 11:24:00 UTC,0.000000,0.000000,0.000000,0.000000,3
15,2013-11-23 12:57:00.000000190,5.0,2013-11-23 12:57:00 UTC,0.000000,0.000000,0.000000,0.000000,1
26,2011-02-07 20:01:00.000000114,6.5,2011-02-07 20:01:00 UTC,0.000000,0.000000,0.000000,0.000000,1
124,2013-01-17 17:22:00.00000043,8.0,2013-01-17 17:22:00 UTC,0.000000,0.000000,0.000000,0.000000,2
192,2010-09-05 17:08:00.00000092,3.7,2010-09-05 17:08:00 UTC,0.000000,0.000000,0.000000,0.000000,5
233,2011-07-24 01:14:35.0000002,8.5,2011-07-24 01:14:35 UTC,0.000000,0.000000,0.000000,0.000000,2
273,2009-10-30 18:13:00.00000021,8.1,2009-10-30 18:13:00 UTC,0.000000,0.000000,0.000000,0.000000,4
314,2015-06-02 23:16:15.00000012,34.0,2015-06-02 23:16:15 UTC,-73.974899,40.751095,-73.908546,40.881878,0
357,2013-07-04 16:41:27.0000002,8.5,2013-07-04 16:41:27 UTC,0.000000,0.000000,0.000000,0.000000,1
376,2014-05-29 05:57:22.0000001,2.5,2014-05-29 05:57:22 UTC,0.000000,0.000000,0.000000,0.000000,1



Summary of invalid rows:


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2479.000000,2479.000000,2479.000000,2479.000000,2479.000000,2479.000000
mean,11.444937,-14.257541,7.000855,-14.149769,7.169498,1.411860
std,10.495099,33.700587,21.278738,30.548428,21.393349,1.334719
min,-44.900000,-736.550000,-74.007670,-84.654241,-74.006377,0.000000
25%,5.700000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,8.100000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,12.900000,0.000000,0.000000,0.000000,0.000000,1.000000
max,128.830000,40.787575,401.083332,40.851027,404.616667,6.000000


In [9]:
# ============================================================
# STEP 9 — SCAN ALL 55.4M ROWS WITHOUT LOADING THEM AT ONCE
# ============================================================
# We process 1 million rows at a time.
# This gives us statistics for the ENTIRE dataset while keeping
# RAM usage manageable.
#
# Important:
# We are ONLY counting problems here.
# We are NOT deleting or modifying any data yet.
# ============================================================

import pandas as pd

total_rows = 0

invalid_counts = {
    "negative_fare": 0,
    "invalid_pickup_longitude": 0,
    "invalid_pickup_latitude": 0,
    "invalid_dropoff_longitude": 0,
    "invalid_dropoff_latitude": 0,
    "invalid_passenger_count": 0,
    "any_invalid": 0
}

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    negative_fare = chunk["fare_amount"] <= 0

    invalid_pickup_longitude = ~chunk["pickup_longitude"].between(-75, -72)
    invalid_pickup_latitude = ~chunk["pickup_latitude"].between(40, 42)

    invalid_dropoff_longitude = ~chunk["dropoff_longitude"].between(-75, -72)
    invalid_dropoff_latitude = ~chunk["dropoff_latitude"].between(40, 42)

    invalid_passenger_count = chunk["passenger_count"] <= 0

    any_invalid = (
        negative_fare
        | invalid_pickup_longitude
        | invalid_pickup_latitude
        | invalid_dropoff_longitude
        | invalid_dropoff_latitude
        | invalid_passenger_count
    )

    invalid_counts["negative_fare"] += negative_fare.sum()
    invalid_counts["invalid_pickup_longitude"] += invalid_pickup_longitude.sum()
    invalid_counts["invalid_pickup_latitude"] += invalid_pickup_latitude.sum()
    invalid_counts["invalid_dropoff_longitude"] += invalid_dropoff_longitude.sum()
    invalid_counts["invalid_dropoff_latitude"] += invalid_dropoff_latitude.sum()
    invalid_counts["invalid_passenger_count"] += invalid_passenger_count.sum()
    invalid_counts["any_invalid"] += any_invalid.sum()

    print(f"Processed {total_rows:,} rows")

print("\n========== FULL DATASET RESULTS ==========")
print(f"Total rows: {total_rows:,}")

for name, count in invalid_counts.items():
    print(f"{name:30s}: {count:,} ({count / total_rows * 100:.2f}%)")

Processed 1,000,000 rows
Processed 2,000,000 rows
Processed 3,000,000 rows
Processed 4,000,000 rows
Processed 5,000,000 rows
Processed 6,000,000 rows
Processed 7,000,000 rows
Processed 8,000,000 rows
Processed 9,000,000 rows
Processed 10,000,000 rows
Processed 11,000,000 rows
Processed 12,000,000 rows
Processed 13,000,000 rows
Processed 14,000,000 rows
Processed 15,000,000 rows
Processed 16,000,000 rows
Processed 17,000,000 rows
Processed 18,000,000 rows
Processed 19,000,000 rows
Processed 20,000,000 rows
Processed 21,000,000 rows
Processed 22,000,000 rows
Processed 23,000,000 rows
Processed 24,000,000 rows
Processed 25,000,000 rows
Processed 26,000,000 rows
Processed 27,000,000 rows
Processed 28,000,000 rows
Processed 29,000,000 rows
Processed 30,000,000 rows
Processed 31,000,000 rows
Processed 32,000,000 rows
Processed 33,000,000 rows
Processed 34,000,000 rows
Processed 35,000,000 rows
Processed 36,000,000 rows
Processed 37,000,000 rows
Processed 38,000,000 rows
Processed 39,000,000 

In [10]:

zero_coordinate_rows = 0
total_rows = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    zero_coords = (
        (chunk["pickup_longitude"] == 0)
        & (chunk["pickup_latitude"] == 0)
        & (chunk["dropoff_longitude"] == 0)
        & (chunk["dropoff_latitude"] == 0)
    )

    zero_coordinate_rows += zero_coords.sum()

print(f"Rows with all coordinates = 0,0: {zero_coordinate_rows:,}")
print(f"Percentage: {zero_coordinate_rows / total_rows * 100:.2f}%")

Rows with all coordinates = 0,0: 1,003,352
Percentage: 1.81%


In [11]:
nonzero_invalid_coords = 0
total_rows = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    invalid_coords = (
        ~chunk["pickup_longitude"].between(-75, -72)
        | ~chunk["pickup_latitude"].between(40, 42)
        | ~chunk["dropoff_longitude"].between(-75, -72)
        | ~chunk["dropoff_latitude"].between(40, 42)
    )

    all_zero = (
        (chunk["pickup_longitude"] == 0)
        & (chunk["pickup_latitude"] == 0)
        & (chunk["dropoff_longitude"] == 0)
        & (chunk["dropoff_latitude"] == 0)
    )

    nonzero_invalid_coords += (invalid_coords & ~all_zero).sum()

print(f"Non-(0,0) invalid-coordinate rows: {nonzero_invalid_coords:,}")
print(f"Percentage: {nonzero_invalid_coords / total_rows * 100:.2f}%")

Non-(0,0) invalid-coordinate rows: 163,203
Percentage: 0.29%


In [12]:
import os
import gc
import numpy as np
import pandas as pd

PROCESSED_DIR = "/kaggle/working/taxi_processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

CHUNK_SIZE = 1_000_000

chunk_number = 0
total_input = 0
total_removed = 0


for chunk in pd.read_csv(TRAIN_PATH, chunksize=CHUNK_SIZE):

    chunk_number += 1
    total_input += len(chunk)

    # --------------------------------------------------------
    # 1. Remove clearly invalid records
    # --------------------------------------------------------
    # These rules were verified against the FULL dataset.
    #
    # Coordinates must fall within a reasonable NYC bounding
    # box. Passenger count must be positive and fare must be
    # positive.
    # --------------------------------------------------------

    valid_mask = (
        chunk["fare_amount"].gt(0)
        & chunk["pickup_longitude"].between(-75, -72)
        & chunk["pickup_latitude"].between(40, 42)
        & chunk["dropoff_longitude"].between(-75, -72)
        & chunk["dropoff_latitude"].between(40, 42)
        & chunk["passenger_count"].gt(0)
    )

    removed = (~valid_mask).sum()
    total_removed += removed

    chunk = chunk.loc[valid_mask].copy()

    # --------------------------------------------------------
    # 2. Parse datetime
    # --------------------------------------------------------
    # The raw datetime string is not directly useful to most
    # ML models. We convert it into numerical time features.
    # --------------------------------------------------------

    dt = pd.to_datetime(chunk["pickup_datetime"], errors="coerce")

    chunk["year"] = dt.dt.year.astype("int16")
    chunk["month"] = dt.dt.month.astype("int8")
    chunk["hour"] = dt.dt.hour.astype("int8")
    chunk["day_of_week"] = dt.dt.dayofweek.astype("int8")

    # --------------------------------------------------------
    # 3. Calculate geographic distance
    # --------------------------------------------------------
    # Latitude/longitude describe the trip endpoints.
    # Haversine distance gives us an approximate straight-line
    # distance between pickup and dropoff.
    # --------------------------------------------------------

    lat1 = np.radians(chunk["pickup_latitude"].astype("float32"))
    lat2 = np.radians(chunk["dropoff_latitude"].astype("float32"))

    dlat = lat2 - lat1

    lon1 = np.radians(chunk["pickup_longitude"].astype("float32"))
    lon2 = np.radians(chunk["dropoff_longitude"].astype("float32"))

    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    chunk["distance_km"] = (
        6371 * 2 * np.arcsin(np.sqrt(a))
    ).astype("float32")

    # --------------------------------------------------------
    # 4. Remove columns we no longer need
    # --------------------------------------------------------
    # 'key' is an identifier and the raw datetime has already
    # been converted into useful numerical features.
    # The original coordinates are also removed because their
    # information is represented by distance.
    # --------------------------------------------------------

    chunk.drop(
        columns=[
            "key",
            "pickup_datetime",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude"
        ],
        inplace=True
    )

    # --------------------------------------------------------
    # 5. Downcast numerical columns
    # --------------------------------------------------------
    # Smaller dtypes reduce RAM usage and storage size.
    # float32 is sufficient for these features; passenger count
    # and extracted time features need only small integers.
    # --------------------------------------------------------

    chunk["fare_amount"] = chunk["fare_amount"].astype("float32")
    chunk["passenger_count"] = chunk["passenger_count"].astype("int8")

    # --------------------------------------------------------
    # 6. Save this processed chunk as Parquet
    # --------------------------------------------------------
    # We save separate Parquet files instead of concatenating
    # 55M rows into one giant DataFrame.
    # --------------------------------------------------------

    output_path = os.path.join(
        PROCESSED_DIR,
        f"part_{chunk_number:03d}.parquet"
    )

    chunk.to_parquet(output_path, index=False)

    print(
        f"Chunk {chunk_number:02d} | "
        f"Input: {total_input:,} | "
        f"Removed: {removed:,} | "
        f"Saved: {len(chunk):,}"
    )

    # Explicitly release memory before reading the next chunk.
    del chunk, dt, lat1, lat2, dlat, lon1, lon2, dlon, a
    gc.collect()


print("\n========== PROCESSING COMPLETE ==========")
print(f"Total input rows : {total_input:,}")
print(f"Total removed    : {total_removed:,}")
print(f"Output directory : {PROCESSED_DIR}")

Chunk 01 | Input: 1,000,000 | Removed: 24,322 | Saved: 975,678
Chunk 02 | Input: 2,000,000 | Removed: 24,430 | Saved: 975,570
Chunk 03 | Input: 3,000,000 | Removed: 24,837 | Saved: 975,163
Chunk 04 | Input: 4,000,000 | Removed: 24,705 | Saved: 975,295
Chunk 05 | Input: 5,000,000 | Removed: 24,562 | Saved: 975,438
Chunk 06 | Input: 6,000,000 | Removed: 24,417 | Saved: 975,583
Chunk 07 | Input: 7,000,000 | Removed: 24,529 | Saved: 975,471
Chunk 08 | Input: 8,000,000 | Removed: 24,663 | Saved: 975,337
Chunk 09 | Input: 9,000,000 | Removed: 24,769 | Saved: 975,231
Chunk 10 | Input: 10,000,000 | Removed: 24,786 | Saved: 975,214
Chunk 11 | Input: 11,000,000 | Removed: 24,428 | Saved: 975,572
Chunk 12 | Input: 12,000,000 | Removed: 24,391 | Saved: 975,609
Chunk 13 | Input: 13,000,000 | Removed: 24,667 | Saved: 975,333
Chunk 14 | Input: 14,000,000 | Removed: 24,403 | Saved: 975,597
Chunk 15 | Input: 15,000,000 | Removed: 24,275 | Saved: 975,725
Chunk 16 | Input: 16,000,000 | Removed: 24,765 | 

In [13]:
import os
import glob
import pandas as pd

parquet_files = sorted(
    glob.glob("/kaggle/working/taxi_processed/*.parquet")
)

print("Number of Parquet files:", len(parquet_files))

# Read only the first processed chunk.
check_df = pd.read_parquet(parquet_files[0])

print("\nShape:", check_df.shape)

print("\nColumns:")
print(check_df.columns.tolist())

print("\nDtypes:")
print(check_df.dtypes)

print("\nMemory usage:")
print(f"{check_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nFirst 5 rows:")
display(check_df.head())

# Calculate total size of the processed Parquet dataset.
total_size_gb = sum(
    os.path.getsize(file)
    for file in parquet_files
) / (1024 ** 3)

print(f"\nTotal Parquet size: {total_size_gb:.3f} GB")

Number of Parquet files: 56

Shape: (975678, 7)

Columns:
['fare_amount', 'passenger_count', 'year', 'month', 'hour', 'day_of_week', 'distance_km']

Dtypes:
fare_amount        float32
passenger_count       int8
year                 int16
month                 int8
hour                  int8
day_of_week           int8
distance_km        float32
dtype: object

Memory usage:
13.03 MB

First 5 rows:


,fare_amount,passenger_count,year,month,hour,day_of_week,distance_km
0,4.5,1,2009,6,17,0,1.031069
1,16.9,1,2010,1,16,1,8.449763
2,5.7,2,2011,8,0,3,1.389644
3,7.7,1,2012,4,4,5,2.799485
4,5.3,1,2010,3,7,1,1.998886



Total Parquet size: 0.414 GB


In [14]:
test_df = pd.read_csv(
    "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/test.csv"
)

test_df["pickup_datetime"] = pd.to_datetime(
    test_df["pickup_datetime"],
    errors="coerce"
)

print("TEST PERIOD:")
print("Minimum:", test_df["pickup_datetime"].min())
print("Maximum:", test_df["pickup_datetime"].max())

print("\nTEST rows:", len(test_df))

# Count training rows by year using the already-processed
# Parquet files. We don't need to touch the original 5.3 GB CSV.
year_counts = {}

for file in parquet_files:
    part = pd.read_parquet(file, columns=["year"])

    counts = part["year"].value_counts()

    for year, count in counts.items():
        year_counts[year] = year_counts.get(year, 0) + count

print("\nTRAINING ROWS BY YEAR:")
for year, count in sorted(year_counts.items()):
    print(f"{year}: {count:,}")

TEST PERIOD:
Minimum: 2009-01-01 11:04:24+00:00
Maximum: 2015-06-30 20:03:50+00:00

TEST rows: 9914

TRAINING ROWS BY YEAR:
2009: 8,434,051
2010: 8,170,323
2011: 8,494,153
2012: 8,632,059
2013: 8,479,959
2014: 8,072,536
2015: 3,780,053


In [15]:
processed_rows = 0

for file in parquet_files:
    processed_rows += len(
        pd.read_parquet(file, columns=["fare_amount"])
    )

print(f"Processed rows: {processed_rows:,}")

Processed rows: 54,063,134


In [16]:
import os
import gc
import numpy as np
import pandas as pd

VALIDATION_FRACTION = 0.10
VALIDATION_SEED = 42

VALIDATION_DIR = "/kaggle/working/taxi_validation"
TRAIN_POOL_DIR = "/kaggle/working/taxi_train_pool"

os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(TRAIN_POOL_DIR, exist_ok=True)

global_row_start = 0
validation_rows = 0
training_rows = 0

for file_number, file in enumerate(parquet_files, start=1):

    # --------------------------------------------------------
    # Read one Parquet chunk at a time.
    # --------------------------------------------------------
    part = pd.read_parquet(file)

    n = len(part)

    # --------------------------------------------------------
    # Create deterministic random numbers for these rows.
    #
    # The seed depends on the global row positions, so the same
    # row will always receive the same train/validation decision.
    # --------------------------------------------------------

    rng = np.random.default_rng(
        VALIDATION_SEED + file_number
    )

    validation_mask = (
        rng.random(n) < VALIDATION_FRACTION
    )

    validation_part = part.loc[validation_mask]
    training_part = part.loc[~validation_mask]

    # --------------------------------------------------------
    # Save validation and training-pool pieces separately.
    # --------------------------------------------------------

    validation_path = os.path.join(
        VALIDATION_DIR,
        f"validation_{file_number:03d}.parquet"
    )

    training_path = os.path.join(
        TRAIN_POOL_DIR,
        f"train_{file_number:03d}.parquet"
    )

    validation_part.to_parquet(
        validation_path,
        index=False
    )

    training_part.to_parquet(
        training_path,
        index=False
    )

    validation_rows += len(validation_part)
    training_rows += len(training_part)

    print(
        f"Part {file_number:02d} | "
        f"Train: {len(training_part):,} | "
        f"Validation: {len(validation_part):,}"
    )

    del part, validation_part, training_part
    gc.collect()


print("\n========== SPLIT COMPLETE ==========")
print(f"Training pool : {training_rows:,}")
print(f"Validation    : {validation_rows:,}")
print(f"Total         : {training_rows + validation_rows:,}")
print(
    f"Validation %  : "
    f"{validation_rows / (training_rows + validation_rows) * 100:.2f}%"
)

Part 01 | Train: 878,107 | Validation: 97,571
Part 02 | Train: 877,724 | Validation: 97,846
Part 03 | Train: 877,589 | Validation: 97,574
Part 04 | Train: 878,135 | Validation: 97,160
Part 05 | Train: 877,623 | Validation: 97,815
Part 06 | Train: 878,433 | Validation: 97,150
Part 07 | Train: 877,748 | Validation: 97,723
Part 08 | Train: 877,850 | Validation: 97,487
Part 09 | Train: 877,953 | Validation: 97,278
Part 10 | Train: 877,300 | Validation: 97,914
Part 11 | Train: 878,111 | Validation: 97,461
Part 12 | Train: 878,368 | Validation: 97,241
Part 13 | Train: 877,957 | Validation: 97,376
Part 14 | Train: 878,222 | Validation: 97,375
Part 15 | Train: 877,672 | Validation: 98,053
Part 16 | Train: 877,314 | Validation: 97,921
Part 17 | Train: 877,865 | Validation: 97,510
Part 18 | Train: 878,408 | Validation: 97,191
Part 19 | Train: 877,822 | Validation: 97,640
Part 20 | Train: 877,761 | Validation: 97,512
Part 21 | Train: 877,504 | Validation: 97,704
Part 22 | Train: 877,891 | Validat

In [17]:
import pandas as pd
import numpy as np
import glob

train_pool_files = sorted(
    glob.glob("/kaggle/working/taxi_train_pool/*.parquet")
)

validation_files = sorted(
    glob.glob("/kaggle/working/taxi_validation/*.parquet")
)

# ------------------------------------------------------------
# Collect a small sample from each side.
#
# We do NOT need millions of rows just to check whether the
# distributions look reasonable.
# ------------------------------------------------------------

train_samples = []
validation_samples = []

for train_file, val_file in zip(train_pool_files, validation_files):

    train_part = pd.read_parquet(
        train_file,
        columns=[
            "fare_amount",
            "distance_km",
            "year",
            "passenger_count"
        ]
    )

    val_part = pd.read_parquet(
        val_file,
        columns=[
            "fare_amount",
            "distance_km",
            "year",
            "passenger_count"
        ]
    )

    # Take a small random sample from each chunk.
    train_samples.append(
        train_part.sample(
            n=min(5_000, len(train_part)),
            random_state=42
        )
    )

    validation_samples.append(
        val_part.sample(
            n=min(5_000, len(val_part)),
            random_state=42
        )
    )


train_check = pd.concat(
    train_samples,
    ignore_index=True
)

validation_check = pd.concat(
    validation_samples,
    ignore_index=True
)

# ------------------------------------------------------------
# Compare numerical distributions.
# ------------------------------------------------------------

print("========== TRAIN vs VALIDATION ==========\n")

comparison = pd.DataFrame({
    "Train": train_check[
        ["fare_amount", "distance_km", "passenger_count"]
    ].mean(),

    "Validation": validation_check[
        ["fare_amount", "distance_km", "passenger_count"]
    ].mean()
})

print(comparison)

# ------------------------------------------------------------
# Compare year distribution.
# ------------------------------------------------------------

print("\n========== YEAR DISTRIBUTION ==========\n")

train_year_pct = (
    train_check["year"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

val_year_pct = (
    validation_check["year"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

year_comparison = pd.DataFrame({
    "Train %": train_year_pct,
    "Validation %": val_year_pct
}).fillna(0)

print(year_comparison.round(2))

# ------------------------------------------------------------
# Compare target quantiles.
#
# This is particularly important because fare_amount is our
# regression target.
# ------------------------------------------------------------

print("\n========== FARE QUANTILES ==========\n")

quantiles = [0.01, 0.25, 0.50, 0.75, 0.99]

fare_comparison = pd.DataFrame({
    "Train": train_check["fare_amount"].quantile(quantiles),
    "Validation": validation_check["fare_amount"].quantile(quantiles)
})

print(fare_comparison)

print("\n========== CHECK COMPLETE ==========")

========== TRAIN vs VALIDATION ==========

                     Train  Validation
fare_amount      11.336082   11.319543
distance_km       3.330868    3.320356
passenger_count   1.687696    1.692918

========== YEAR DISTRIBUTION ==========

      Train %  Validation %
year                       
2009    15.56         15.66
2010    15.17         15.06
2011    15.75         15.74
2012    15.94         15.92
2013    15.69         15.70
2014    14.92         14.87
2015     6.96          7.05

========== FARE QUANTILES ==========

      Train  Validation
0.01    3.3    3.300000
0.25    6.0    6.000000
0.50    8.5    8.500000
0.75   12.5   12.500000
0.99   52.0   52.830002

========== CHECK COMPLETE ==========


In [18]:
import os
import glob
import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TARGET_TRAIN_ROWS = 1_000_000
RANDOM_STATE = 42

TRAIN_POOL_DIR = "/kaggle/working/taxi_train_pool"

train_pool_files = sorted(
    glob.glob(os.path.join(TRAIN_POOL_DIR, "*.parquet"))
)

# These are the features currently available in our processed
# Parquet files.
FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

file_sizes = []

for file in train_pool_files:
    # Reading only the target column gives us the row count
    # while avoiding unnecessary columns.
    n_rows = len(
        pd.read_parquet(
            file,
            columns=[TARGET]
        )
    )

    file_sizes.append(n_rows)

total_training_rows = sum(file_sizes)

print(f"Training pool rows: {total_training_rows:,}")
print(f"Target sample size: {TARGET_TRAIN_ROWS:,}")

assert total_training_rows == 48_655_661, (
    "Training-pool row count does not match our fixed split."
)

raw_allocations = (
    np.array(file_sizes) / total_training_rows
) * TARGET_TRAIN_ROWS

sample_sizes = np.floor(raw_allocations).astype(int)

# Distribute the remaining rows caused by rounding.
remaining = TARGET_TRAIN_ROWS - sample_sizes.sum()

if remaining > 0:
    fractional_parts = raw_allocations - sample_sizes

    # Give the remaining rows to files with the largest
    # fractional remainders.
    largest_remainders = np.argsort(
        fractional_parts
    )[::-1][:remaining]

    sample_sizes[largest_remainders] += 1

assert sample_sizes.sum() == TARGET_TRAIN_ROWS

# ------------------------------------------------------------
# Sample each Parquet file.
# ------------------------------------------------------------

sample_parts = []

for i, (file, n_sample) in enumerate(
    zip(train_pool_files, sample_sizes)
):

    if n_sample == 0:
        continue

    # Read only the columns required for modeling.
    part = pd.read_parquet(
        file,
        columns=FEATURES + [TARGET]
    )

    # Use a deterministic seed for every file.
    # This makes the 1M sample reproducible.
    sampled_part = part.sample(
        n=n_sample,
        random_state=RANDOM_STATE + i
    )

    sample_parts.append(sampled_part)

    print(
        f"Part {i + 1:02d}: "
        f"{n_sample:,} sampled rows"
    )

    del part, sampled_part
    gc.collect()

train_1m = pd.concat(
    sample_parts,
    ignore_index=True
)

train_1m = train_1m.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


assert len(train_1m) == TARGET_TRAIN_ROWS
assert train_1m[TARGET].notna().all()

print("\n========== 1M SAMPLE READY ==========")
print(f"Shape: {train_1m.shape}")
print(f"Rows : {len(train_1m):,}")
print(f"Memory: {train_1m.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

display(train_1m.head())

Training pool rows: 48,655,661
Target sample size: 1,000,000
Part 01: 18,047 sampled rows
Part 02: 18,040 sampled rows
Part 03: 18,037 sampled rows
Part 04: 18,048 sampled rows
Part 05: 18,037 sampled rows
Part 06: 18,054 sampled rows
Part 07: 18,040 sampled rows
Part 08: 18,042 sampled rows
Part 09: 18,044 sampled rows
Part 10: 18,031 sampled rows
Part 11: 18,047 sampled rows
Part 12: 18,053 sampled rows
Part 13: 18,044 sampled rows
Part 14: 18,050 sampled rows
Part 15: 18,038 sampled rows
Part 16: 18,031 sampled rows
Part 17: 18,042 sampled rows
Part 18: 18,054 sampled rows
Part 19: 18,042 sampled rows
Part 20: 18,040 sampled rows
Part 21: 18,035 sampled rows
Part 22: 18,043 sampled rows
Part 23: 18,055 sampled rows
Part 24: 18,041 sampled rows
Part 25: 18,033 sampled rows
Part 26: 18,043 sampled rows
Part 27: 18,036 sampled rows
Part 28: 18,045 sampled rows
Part 29: 18,032 sampled rows
Part 30: 18,045 sampled rows
Part 31: 18,037 sampled rows
Part 32: 18,048 sampled rows
Part 33: 18

,passenger_count,year,month,hour,day_of_week,distance_km,fare_amount
0,1,2009,8,19,4,1.729130,6.5
1,1,2011,11,10,0,0.715955,4.9
2,2,2015,1,20,5,3.654863,10.0
3,1,2011,3,4,6,1.912334,6.1
4,2,2013,9,18,5,5.252647,24.0


In [19]:
import os
import glob
import gc
import time

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import mean_squared_error

FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

VALIDATION_DIR = "/kaggle/working/taxi_validation"

validation_files = sorted(
    glob.glob(
        os.path.join(
            VALIDATION_DIR,
            "*.parquet"
        )
    )
)


print("Loading fixed validation set...")

validation_parts = []

for file in validation_files:

    part = pd.read_parquet(
        file,
        columns=FEATURES + [TARGET]
    )

    validation_parts.append(part)

validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)

del validation_parts
gc.collect()

print(
    f"Validation shape: {validation_df.shape}"
)


X_train_1m = train_1m[FEATURES]
y_train_1m = train_1m[TARGET]

X_val = validation_df[FEATURES]
y_val = validation_df[TARGET]


dtrain = xgb.DMatrix(
    X_train_1m,
    label=y_train_1m
)

dval = xgb.DMatrix(
    X_val,
    label=y_val
)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",

    # Tree complexity
    "max_depth": 8,
    "min_child_weight": 10,

    # Learning
    "eta": 0.10,

    # Randomization / regularization
    "subsample": 0.8,
    "colsample_bytree": 0.8,

    # Parallel CPU training
    "tree_method": "hist",
    "nthread": -1,

    "seed": 42
}

print("\n========== TRAINING XGBOOST ==========")

start_time = time.time()

xgb_1m_model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=500,

    evals=[
        (dtrain, "train"),
        (dval, "validation")
    ],

    early_stopping_rounds=30,
    verbose_eval=25
)

training_time = time.time() - start_time


print("\n========== EVALUATION ==========")

val_predictions = xgb_1m_model.predict(
    dval
)

rmse_1m = np.sqrt(
    mean_squared_error(
        y_val,
        val_predictions
    )
)

print(f"1M Training RMSE : {rmse_1m:.6f}")
print(
    f"Best boosting round: "
    f"{xgb_1m_model.best_iteration}"
)
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

print("\n========== BASELINE COMPLETE ==========")



Loading fixed validation set...
Validation shape: (5407473, 7)

========== TRAINING XGBOOST ==========
[0]	train-rmse:8.96205	validation-rmse:33.78560
[25]	train-rmse:4.89731	validation-rmse:32.94387
[50]	train-rmse:4.51127	validation-rmse:32.89196
[75]	train-rmse:4.47469	validation-rmse:32.88995
[100]	train-rmse:4.44807	validation-rmse:32.88967
[120]	train-rmse:4.42929	validation-rmse:32.89022

========== EVALUATION ==========
1M Training RMSE : 32.890219
Best boosting round: 90
Training time: 0.41 minutes

========== BASELINE COMPLETE ==========


In [20]:
import time
import gc
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_error

FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

# ------------------------------------------------------------
# Prepare training data
# ------------------------------------------------------------

X_train_1m_lgb = train_1m[FEATURES]
y_train_1m_lgb = train_1m[TARGET]

# We already loaded the fixed validation set earlier.
X_val_lgb = validation_df[FEATURES]
y_val_lgb = validation_df[TARGET]

# ------------------------------------------------------------
# LightGBM datasets
# ------------------------------------------------------------

lgb_train = lgb.Dataset(
    X_train_1m_lgb,
    label=y_train_1m_lgb,
    free_raw_data=False
)

lgb_val = lgb.Dataset(
    X_val_lgb,
    label=y_val_lgb,
    reference=lgb_train,
    free_raw_data=False
)

# ------------------------------------------------------------
# Baseline parameters
# ------------------------------------------------------------

lgb_params = {
    "objective": "regression",
    "metric": "rmse",

    # Tree complexity
    "num_leaves": 64,
    "max_depth": -1,
    "min_data_in_leaf": 100,

    # Learning
    "learning_rate": 0.05,

    # Randomization / regularization
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Reproducibility / CPU
    "seed": 42,
    "verbosity": -1,
    "n_jobs": -1
}

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("========== TRAINING LIGHTGBM ==========")

start_time = time.time()

lgb_1m_model = lgb.train(
    params=lgb_params,
    train_set=lgb_train,
    num_boost_round=2000,

    valid_sets=[
        lgb_train,
        lgb_val
    ],

    valid_names=[
        "train",
        "validation"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# Predict on the SAME fixed validation set
# ------------------------------------------------------------

print("\n========== EVALUATION ==========")

lgb_predictions = lgb_1m_model.predict(
    X_val_lgb,
    num_iteration=lgb_1m_model.best_iteration
)

rmse_lgb_1m = np.sqrt(
    mean_squared_error(
        y_val_lgb,
        lgb_predictions
    )
)

print(f"1M LightGBM RMSE : {rmse_lgb_1m:.6f}")
print(
    f"Best boosting round: "
    f"{lgb_1m_model.best_iteration}"
)
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

# ------------------------------------------------------------
# Compare directly with our XGBoost baseline
# ------------------------------------------------------------

print("\n========== MODEL COMPARISON ==========")

print(f"XGBoost  : {rmse_1m:.6f}")
print(f"LightGBM : {rmse_lgb_1m:.6f}")

if rmse_lgb_1m < rmse_1m:
    print("LightGBM currently has the lower RMSE.")
else:
    print("XGBoost currently has the lower RMSE.")

print("\n========== LIGHTGBM BASELINE COMPLETE ==========")

# Free temporary LightGBM dataset objects.
del lgb_train, lgb_val
gc.collect()

========== TRAINING LIGHTGBM ==========
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[184]	train's rmse: 4.4722	validation's rmse: 32.8865

========== EVALUATION ==========
1M LightGBM RMSE : 32.886544
Best boosting round: 184
Training time: 0.62 minutes

========== MODEL COMPARISON ==========
XGBoost  : 32.890219
LightGBM : 32.886544
LightGBM currently has the lower RMSE.

========== LIGHTGBM BASELINE COMPLETE ==========


215